# Utils - QB - FrequencyNormalizer

Ce notebook illustre et vérifie le comportement de la classe `FrequencyNormalizer`
(`tsforecast/utils/frequency/normalizer.py`), qui centralise la normalisation des
représentations de fréquences (codes pandas-like `'D'`, `'M'`, `'Q'`... vs noms
littéraux `'daily'`, `'monthly'`, `'quarterly'`...).

**Hors périmètre** : cette classe ne connaît ni `full_periods_only`, ni
`limit_direction`, ni `limit_area`, ni `target_position` — ces paramètres
appartiennent à la mécanique de conversion proprement dite (`FrequencyConverter`,
qui utilise `FrequencyNormalizer` en interne pour normaliser ses fréquences), déjà
testée en détail dans `frequency_converter.ipynb`. Ce notebook-ci se concentre
uniquement sur la normalisation des représentations de fréquence.

On se concentre ici sur les **méthodes publiques** de `FrequencyNormalizer`
(`normalize`, `to_code`, `to_literal`, `to_pandas_freq`, `to_dateoffset`,
`validate`, `is_higher_frequency`, `are_compatible_frequencies`) ainsi que sur les
**fonctions publiques** de `tsforecast/utils/frequency/utils.py` qui les exposent
au niveau module (`normalize_frequency`, `to_literal`, `to_code`, `to_pandas_freq`,
`to_dateoffset`, `is_higher_frequency`, `validate_frequency`, `get_frequency_order`).
`convert_frequency()` (également exposée par `utils.py`) délègue entièrement à
`FrequencyConverter` : elle est hors périmètre ici, voir `frequency_converter.ipynb`.
Les méthodes/fonctions privées (préfixées `_`) et la classe abstraite parente
`TemporalNormalizer` ne sont pas testées directement.

**Jeux de données** : `FrequencyNormalizer` opère sur des chaînes de fréquence, pas
sur des séries temporelles à proprement parler. On l'illustre donc principalement
avec des chaînes ciblées couvrant les cas limites (casse, positions S/E, ancrages,
types invalides), complétées en section 10 par des index pandas réels détectés via
`detect_index_frequency` et par les indicateurs macroéconomiques (fréquences de
publication) définis dans `3 - QB - Panel a frequences mixtes heterogene.ipynb` et
`duration_normalizer.ipynb`.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/utils/frequency/test_normalizer.py`, qui n'existe pas encore).

## 1 - Import et instanciation

In [1]:
# Importation des modules
import warnings
from typing import get_args

import pandas as pd

# Classe testée
from tsforecast.utils.frequency.normalizer import FrequencyNormalizer

# Types exportés (pour lister les fréquences déclarées, à titre indicatif)
from tsforecast.utils.frequency.types import FrequencyType, UserFrequencyType

warnings.filterwarnings('ignore')

# Instanciation du normalizer
normalizer = FrequencyNormalizer()

codes_declares = list(get_args(FrequencyType))
litteraux_declares = list(get_args(UserFrequencyType))

print("Codes de fréquence déclarés dans FrequencyType :")
print(codes_declares)
print()
print("Noms littéraux déclarés dans UserFrequencyType :")
print(litteraux_declares)

Codes de fréquence déclarés dans FrequencyType :
['ns', 'us', 'ms', 's', 'min', 'T', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'A', 'Y']

Noms littéraux déclarés dans UserFrequencyType :
['daily', 'weekly', 'monthly', 'quarterly', 'annual', 'business_daily']


### 1.1 - Piège : `FrequencyType`/`UserFrequencyType` déclarent des valeurs que la classe ne supporte pas réellement

Les types `Literal` ci-dessus servent d'indication pour les annotations, mais
`FrequencyNormalizer` construit ses mappings à partir de ses propres dictionnaires
internes (`_pandas_to_literal`, `_literal_to_pandas`), qui ne couvrent pas
exactement les mêmes ensembles. On les compare ci-dessous.

In [2]:
# Codes et littéraux RÉELLEMENT supportés par la classe (source de vérité)
codes_supportes = list(normalizer._pandas_to_literal.keys())
litteraux_supportes = list(normalizer._literal_to_pandas.keys())

print("Codes réellement supportés :", codes_supportes)
print("Littéraux réellement supportés :", litteraux_supportes)

# Codes déclarés dans le type mais absents du mapping réel
codes_fantomes = set(codes_declares) - set(codes_supportes)
print(f"\nCodes déclarés dans FrequencyType mais NON supportés par la classe : {codes_fantomes}")

# Littéraux réellement supportés mais absents du type déclaré (l'inverse)
litteraux_non_declares = set(litteraux_supportes) - set(litteraux_declares)
print(f"Littéraux supportés par la classe mais absents de UserFrequencyType  : {litteraux_non_declares}")

Codes réellement supportés : ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
Littéraux réellement supportés : ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual']

Codes déclarés dans FrequencyType mais NON supportés par la classe : {'A', 'T'}
Littéraux supportés par la classe mais absents de UserFrequencyType  : {'nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'semi_monthly', 'hourly'}


## 2 - `normalize()` : normalisation vers un code

### 2.1 - Codes et littéraux réellement supportés

In [3]:
# Un code supporté est toujours renvoyé inchangé
for c in codes_supportes:
    assert normalizer.normalize(c) == c, c
print("OK : normalize(code) == code pour tous les codes réellement supportés")

# Chaque nom littéral supporté est converti vers son code correspondant
for lit in litteraux_supportes:
    resultat = normalizer.normalize(lit)
    print(f"{lit:15s} -> {resultat}")

OK : normalize(code) == code pour tous les codes réellement supportés
nanosecond      -> ns
microsecond     -> us
millisecond     -> ms
second          -> s
minute          -> min
hourly          -> h
daily           -> D
business_daily  -> B
weekly          -> W
semi_monthly    -> SM
monthly         -> M
quarterly       -> Q
annual          -> Y


### 2.2 - Piège : `'T'` et `'A'`, déclarés dans `FrequencyType`, lèvent `ValueError`

`'T'` (alias pandas historique de la minute) et `'A'` (alias pandas historique de
l'année) figurent dans `FrequencyType` mais pas dans `_pandas_to_literal` : leur
normalisation échoue, y compris via le repli sur `parse_frequency` (la base
extraite est identique à la valeur d'entrée, ce qui bloque la récursion — garde-fou
anti-boucle infinie — sans jamais retomber sur `'min'`/`'Y'`).

In [4]:
for v in codes_fantomes:
    try:
        normalizer.normalize(v)
        print(f"normalize({v!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"normalize({v!r}) -> ValueError : {e}")

# Les codes reconnus pour les mêmes concepts utilisent une casse ou une forme différente
print("\nÉquivalents réellement supportés : 'min' (et non 'T'), 'Y' (et non 'A')")
print("normalize('min') ->", normalizer.normalize('min'))
print("normalize('Y')   ->", normalizer.normalize('Y'))

normalize('A') -> ValueError : Unsupported frequency: A. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
normalize('T') -> ValueError : Unsupported frequency: T. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']

Équivalents réellement supportés : 'min' (et non 'T'), 'Y' (et non 'A')
normalize('min') -> min
normalize('Y')   -> Y


### 2.3 - Sensibilité à la casse (point de vigilance)

`normalize()` fait un lookup exact dans des dictionnaires (`_pandas_to_literal` /
`_literal_to_pandas`) : aucune normalisation de casse n'est appliquée en amont.
Une variante mal capitalisée d'un code ou d'un littéral pourtant valide est donc
rejetée.

In [5]:
# 'D' (code valide) fonctionne, mais 'd' (minuscule) échoue
print("'D' ->", normalizer.normalize('D'))
try:
    normalizer.normalize('d')
except ValueError as e:
    print("'d' -> ValueError :", e)

# Idem pour les littéraux : 'monthly' fonctionne, 'MONTHLY' et 'Monthly' échouent
print("'monthly' ->", normalizer.normalize('monthly'))
for variante in ['MONTHLY', 'Monthly']:
    try:
        normalizer.normalize(variante)
        print(f"{variante!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{variante!r} -> ValueError : {e}")

# 'min' (minute) est un code valide, 'MIN'/'Min' ne le sont pas
print("'min' ->", normalizer.normalize('min'))
for variante in ['MIN', 'Min']:
    try:
        normalizer.normalize(variante)
    except ValueError as e:
        print(f"{variante!r} -> ValueError : {e}")

'D' -> D
'd' -> ValueError : Unsupported frequency: d. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
'monthly' -> M
'MONTHLY' -> ValueError : Unsupported frequency: MONTHLY. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
'Monthly' -> ValueError : Unsupported frequency: Monthly. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q

### 2.4 - Repli sur `parse_frequency` pour les chaînes de fréquence pandas complètes

Si la valeur n'est trouvée ni comme code ni comme littéral, `normalize()` tente
d'extraire une fréquence de base via `parse_frequency()` (`tsforecast/utils/parse/`),
qui décompose une chaîne `[FREQ][S|E]?[-SUFFIXE]?`. Cela permet d'accepter des
chaînes de fréquence pandas complètes (`'MS'`, `'QE-DEC'`...) en ignorant la
position (start/end) et le suffixe d'ancrage.

In [6]:
# La position (S/E) et le suffixe d'ancrage sont ignorés : seule la base compte
exemples = ['MS', 'ME', 'QE-DEC', 'QS-JAN', 'YS-JAN', 'YE-DEC', 'W-MON']
for ex in exemples:
    resultat = normalizer.normalize(ex)
    print(f"{ex:10s} -> {resultat}")

assert normalizer.normalize('MS') == normalizer.normalize('ME') == 'M'
assert normalizer.normalize('QE-DEC') == normalizer.normalize('QS-JAN') == 'Q'
print("OK : la position et le suffixe n'affectent pas le code renvoyé")

MS         -> M
ME         -> M
QE-DEC     -> Q
QS-JAN     -> Q
YS-JAN     -> Y
YE-DEC     -> Y
W-MON      -> W
OK : la position et le suffixe n'affectent pas le code renvoyé


### 2.5 - Erreurs : types non `str` et chaînes non supportées

`normalize()` vérifie explicitement le type de l'entrée et lève toujours un
`ValueError` (jamais un `TypeError`), y compris pour des types manifestement
invalides comme `None`, un entier ou une liste.

In [7]:
for valeur_invalide in [5, None, ['D'], 3.14]:
    try:
        normalizer.normalize(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

# Chaîne str mais non reconnue, même après repli sur parse_frequency
for valeur_invalide in ['xyz', '']:
    try:
        normalizer.normalize(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

5 -> ValueError : Frequency must be a string, got <class 'int'>
None -> ValueError : Frequency must be a string, got <class 'NoneType'>
['D'] -> ValueError : Frequency must be a string, got <class 'list'>
3.14 -> ValueError : Frequency must be a string, got <class 'float'>
'xyz' -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
'' -> ValueError : Unsupported frequency: . Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']


## 3 - `to_code()` : alias explicite de `normalize()`

`to_code()` ne fait qu'appeler `normalize()` en interne : les deux méthodes sont
strictement équivalentes sur toutes les entrées (valides ou non).

In [8]:
for valeur in codes_supportes + litteraux_supportes + ['MS', 'xyz', 'T']:
    try:
        r_normalize = normalizer.normalize(valeur)
    except ValueError as e:
        r_normalize = f'ValueError: {e}'
    try:
        r_to_code = normalizer.to_code(valeur)
    except ValueError as e:
        r_to_code = f'ValueError: {e}'
    assert r_normalize == r_to_code, (valeur, r_normalize, r_to_code)
print("OK : to_code(x) == normalize(x) pour toutes les valeurs testées (y compris les erreurs)")

OK : to_code(x) == normalize(x) pour toutes les valeurs testées (y compris les erreurs)


## 4 - `to_literal()` : conversion vers le nom littéral

### 4.1 - Comportement de base et bijection code <-> littéral

In [9]:
# Conversion code -> littéral et littéral -> littéral (idempotent : normalize puis lookup)
for c in codes_supportes:
    lit = normalizer.to_literal(c)
    assert normalizer.to_literal(lit) == lit
    print(f"{c:4s} -> {lit}")

# Round-trip complet : to_code(to_literal(code)) == code
for c in codes_supportes:
    assert normalizer.to_code(normalizer.to_literal(c)) == c
print("\nOK : to_code(to_literal(code)) == code (bijection code <-> littéral)")

ns   -> nanosecond
us   -> microsecond
ms   -> millisecond
s    -> second
min  -> minute
h    -> hourly
D    -> daily
B    -> business_daily
W    -> weekly
SM   -> semi_monthly
M    -> monthly
Q    -> quarterly
Y    -> annual

OK : to_code(to_literal(code)) == code (bijection code <-> littéral)


### 4.2 - `UserFrequencyType` ne couvre qu'un sous-ensemble des littéraux réellement supportés

Cf. section 1.1 : `to_literal()` peut renvoyer des valeurs comme `'nanosecond'`,
`'minute'` ou `'semi_monthly'`, absentes de `UserFrequencyType` (qui ne déclare que
`daily`/`weekly`/`monthly`/`quarterly`/`annual`/`business_daily`). Le type sert donc
de sous-ensemble indicatif des cas les plus courants, pas d'une liste exhaustive.

In [10]:
for c in ['ns', 'min', 'SM']:
    lit = normalizer.to_literal(c)
    print(f"to_literal({c!r}) = {lit!r:16s} -- déclaré dans UserFrequencyType ? {lit in litteraux_declares}")

to_literal('ns') = 'nanosecond'     -- déclaré dans UserFrequencyType ? False
to_literal('min') = 'minute'         -- déclaré dans UserFrequencyType ? False
to_literal('SM') = 'semi_monthly'   -- déclaré dans UserFrequencyType ? False


### 4.3 - Erreurs

`to_literal()` délègue à `normalize()` : mêmes erreurs (`ValueError`), y compris
pour des types non `str`.

In [11]:
for valeur_invalide in ['xyz', None, 5]:
    try:
        normalizer.to_literal(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

'xyz' -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
None -> ValueError : Frequency must be a string, got <class 'NoneType'>
5 -> ValueError : Frequency must be a string, got <class 'int'>


## 5 - `to_pandas_freq()` : second alias explicite de `normalize()`

Comme `to_code()`, `to_pandas_freq()` ne fait qu'appeler `normalize()` en interne :
les trois méthodes (`normalize`, `to_code`, `to_pandas_freq`) sont strictement
équivalentes sur toutes les entrées. Le nom suggère une spécificité pandas, mais
il n'y a ici aucune différence de comportement.

In [12]:
for valeur in codes_supportes + litteraux_supportes + ['QE-DEC', 'xyz']:
    try:
        r_normalize = normalizer.normalize(valeur)
    except ValueError as e:
        r_normalize = f'ValueError: {e}'
    try:
        r_to_pandas = normalizer.to_pandas_freq(valeur)
    except ValueError as e:
        r_to_pandas = f'ValueError: {e}'
    assert r_normalize == r_to_pandas, (valeur, r_normalize, r_to_pandas)
print("OK : to_pandas_freq(x) == normalize(x) == to_code(x) pour toutes les valeurs testées")

OK : to_pandas_freq(x) == normalize(x) == to_code(x) pour toutes les valeurs testées


## 6 - `validate()` : vérification booléenne, ne lève jamais d'exception

`validate()` encapsule `normalize()` dans un `try/except ValueError` : quelle que
soit l'entrée (chaîne inconnue, type non `str`, valeur `None`...), elle renvoie
toujours un booléen sans jamais propager d'exception. Utile en amont d'un appel à
`normalize()`/`to_code()`/`to_literal()` pour éviter un `try/except` explicite.

In [13]:
# Valeurs valides
for valeur in ['daily', 'D', 'quarterly', 'MS', 'QE-DEC']:
    print(f"validate({valeur!r}) = {normalizer.validate(valeur)}")

print()

# Valeurs invalides de tous types : aucune exception ne remonte, toujours False
# (y compris les codes "fantômes" T/A déclarés dans le type mais non supportés)
for valeur_invalide in ['xyz', '', 'd', 'T', 'A', None, 5, 3.14, ['D'], {'a': 1}]:
    resultat = normalizer.validate(valeur_invalide)
    print(f"validate({valeur_invalide!r}) = {resultat}")
    assert resultat is False

validate('daily') = True
validate('D') = True
validate('quarterly') = True
validate('MS') = True
validate('QE-DEC') = True

validate('xyz') = False
validate('') = False
validate('d') = False
validate('T') = False
validate('A') = False
validate(None) = False
validate(5) = False
validate(3.14) = False
validate(['D']) = False
validate({'a': 1}) = False


## 7 - `to_dateoffset()` : conversion en `pandas.DateOffset`

### 7.1 - Comportement de base

In [14]:
for c in codes_supportes:
    off = normalizer.to_dateoffset(c)
    print(f"{c:4s} -> {off!r} ({type(off).__name__})")

ns   -> <Nano> (Nano)
us   -> <Micro> (Micro)
ms   -> <Milli> (Milli)
s    -> <Second> (Second)
min  -> <Minute> (Minute)
h    -> <Hour> (Hour)
D    -> <Day> (Day)
B    -> <BusinessDay> (BusinessDay)
W    -> <Week: weekday=6> (Week)
SM   -> <SemiMonthEnd: day_of_month=15> (SemiMonthEnd)
M    -> <MonthEnd> (MonthEnd)
Q    -> <QuarterEnd: startingMonth=12> (QuarterEnd)
Y    -> <YearEnd: month=12> (YearEnd)


### 7.2 - Piège : la position (S/E) est perdue avant la création de l'offset

`to_dateoffset()` appelle d'abord `normalize()`, qui ne conserve que la base de
fréquence (cf. section 2.4 : `'MS'`/`'ME'` normalisent tous les deux vers `'M'`).
L'offset construit ensuite via `to_offset('M')` est donc **identique** quelle que
soit la position explicite fournie en entrée — `to_dateoffset()` ne permet pas de
distinguer un ancrage start d'un ancrage end.

In [15]:
offset_ms = normalizer.to_dateoffset('MS')
offset_me = normalizer.to_dateoffset('ME')
offset_m = normalizer.to_dateoffset('M')

print("to_dateoffset('MS') ->", repr(offset_ms))
print("to_dateoffset('ME') ->", repr(offset_me))
print("to_dateoffset('M')  ->", repr(offset_m))
assert offset_ms == offset_me == offset_m
print("\nOK : les trois offsets sont identiques -- la position n'est jamais prise en compte")

to_dateoffset('MS') -> <MonthEnd>
to_dateoffset('ME') -> <MonthEnd>
to_dateoffset('M')  -> <MonthEnd>

OK : les trois offsets sont identiques -- la position n'est jamais prise en compte


### 7.3 - Erreurs

Mêmes erreurs que `normalize()` (délégation directe pour la normalisation), plus
la propagation d'une éventuelle erreur de `pandas.tseries.frequencies.to_offset`
(non observée ici, `_pandas_to_literal` ne contient que des codes valides pour
pandas).

In [16]:
for valeur_invalide in ['xyz', None]:
    try:
        normalizer.to_dateoffset(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

'xyz' -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
None -> ValueError : Frequency must be a string, got <class 'NoneType'>


## 8 - `is_higher_frequency()` : comparaison stricte de deux fréquences

### 8.1 - Comportement de base et formats mixtes (code / littéral)

In [17]:
print("daily > monthly     :", normalizer.is_higher_frequency('daily', 'monthly'))
print("D > M               :", normalizer.is_higher_frequency('D', 'M'))
print("daily > M           :", normalizer.is_higher_frequency('daily', 'M'))
print("monthly > daily     :", normalizer.is_higher_frequency('monthly', 'daily'))
print("quarterly > weekly  :", normalizer.is_higher_frequency('quarterly', 'weekly'))

daily > monthly     : True
D > M               : True
daily > M           : True
monthly > daily     : False
quarterly > weekly  : False


### 8.2 - Égalité : toujours `False` (comparaison stricte `<`, pas `<=`)

In [18]:
# is_higher_frequency(x, x) est toujours False : la comparaison est stricte
for c in codes_supportes:
    assert normalizer.is_higher_frequency(c, c) is False
print("OK : is_higher_frequency(x, x) == False pour tous les codes supportés (comparaison stricte)")

OK : is_higher_frequency(x, x) == False pour tous les codes supportés (comparaison stricte)


### 8.3 - Rang des fréquences "spéciales" : `B` (jour ouvré) et `SM` (semi-mensuel)

`B` (business_daily) et `SM` (semi_monthly) ont un ordre non entier (`7.5` et `8.5`
respectivement dans `_frequency_order`), ce qui les place strictement entre deux
fréquences standards : `D > B > W` (en granularité, `B` est donc une fréquence plus
élevée que `W` mais plus faible que `D`) et `W > SM > M`.

In [19]:
assert normalizer.is_higher_frequency('D', 'B') is True   # jour plus fin que jour ouvré
assert normalizer.is_higher_frequency('B', 'W') is True   # jour ouvré plus fin que semaine
assert normalizer.is_higher_frequency('W', 'SM') is True  # semaine plus fine que semi-mois
assert normalizer.is_higher_frequency('SM', 'M') is True  # semi-mois plus fin que mois
print("OK : D > B > W > SM > M (en granularité décroissante)")

OK : D > B > W > SM > M (en granularité décroissante)


### 8.4 - Matrice complète d'ordre (utile comme table de référence)

In [20]:
matrice = pd.DataFrame(
    {b: [normalizer.is_higher_frequency(a, b) for a in codes_supportes] for b in codes_supportes},
    index=codes_supportes
)
matrice.index.name = 'a \\ b (a plus fin que b ?)'
matrice

,ns,us,ms,s,min,h,D,B,W,SM,M,Q,Y
a \ b (a plus fin que b ?),,,,,,,,,,,,,
ns,False,True,True,True,True,True,True,True,True,True,True,True,True
us,False,False,True,True,True,True,True,True,True,True,True,True,True
ms,False,False,False,True,True,True,True,True,True,True,True,True,True
s,False,False,False,False,True,True,True,True,True,True,True,True,True
min,False,False,False,False,False,True,True,True,True,True,True,True,True
h,False,False,False,False,False,False,True,True,True,True,True,True,True
D,False,False,False,False,False,False,False,True,True,True,True,True,True
B,False,False,False,False,False,False,False,False,True,True,True,True,True
W,False,False,False,False,False,False,False,False,False,True,True,True,True


### 8.5 - Erreur propagée si une fréquence est invalide (pas de repli silencieux sur 0)

`_frequency_order` utilise `.get(code, 0)` en interne, ce qui pourrait laisser
penser qu'une fréquence inconnue est simplement traitée comme "la plus fine"
(ordre `0`). En réalité, `is_higher_frequency()` appelle d'abord `to_code()` sur
les deux arguments, qui lève un `ValueError` **avant** d'atteindre ce `.get()` :
le repli à `0` est donc du code mort avec les mappings actuels (tous les codes
supportés ont une entrée dans `_frequency_order`).

In [21]:
for a, b in [('daily', 'xyz'), ('xyz', 'daily'), ('T', 'D')]:
    try:
        normalizer.is_higher_frequency(a, b)
        print(f"is_higher_frequency({a!r}, {b!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"is_higher_frequency({a!r}, {b!r}) -> ValueError : {e}")

is_higher_frequency('daily', 'xyz') -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
is_higher_frequency('xyz', 'daily') -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
is_higher_frequency('T', 'D') -> ValueError : Unsupported frequency: T. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms',

## 9 - `are_compatible_frequencies()` : à ne pas confondre avec une compatibilité de conversion

Le nom de la méthode suggère une vérification de compatibilité *entre* deux
fréquences (par exemple : est-ce que convertir `freq1` en `freq2` a du sens ?). En
pratique, l'implémentation se contente de normaliser `freq1` puis `freq2`
indépendamment, sans jamais comparer les deux résultats entre eux : elle est donc
strictement équivalente à `validate(freq1) and validate(freq2)`.

In [22]:
# Deux fréquences aux ordres de grandeur radicalement différents sont jugées "compatibles"
# (aucune notion de rapport de conversion raisonnable n'est vérifiée)
print("ns / Y compatibles :", normalizer.are_compatible_frequencies('ns', 'Y'))
print("business_daily / monthly compatibles :", normalizer.are_compatible_frequencies('business_daily', 'monthly'))

# Équivalence stricte avec validate(freq1) and validate(freq2)
paires = [('daily', 'monthly'), ('ns', 'Y'), ('daily', 'xyz'), ('xyz', 'daily'), ('xyz', 'abc'), ('T', 'D')]
for a, b in paires:
    attendu = normalizer.validate(a) and normalizer.validate(b)
    obtenu = normalizer.are_compatible_frequencies(a, b)
    print(f"are_compatible_frequencies({a!r}, {b!r}) = {obtenu}  (== validate(a) and validate(b) : {attendu})")
    assert obtenu == attendu
print("\nOK : are_compatible_frequencies(a, b) == validate(a) and validate(b), sans logique conjointe supplémentaire")

ns / Y compatibles : True
business_daily / monthly compatibles : True
are_compatible_frequencies('daily', 'monthly') = True  (== validate(a) and validate(b) : True)
are_compatible_frequencies('ns', 'Y') = True  (== validate(a) and validate(b) : True)
are_compatible_frequencies('daily', 'xyz') = False  (== validate(a) and validate(b) : False)
are_compatible_frequencies('xyz', 'daily') = False  (== validate(a) and validate(b) : False)
are_compatible_frequencies('xyz', 'abc') = False  (== validate(a) and validate(b) : False)
are_compatible_frequencies('T', 'D') = False  (== validate(a) and validate(b) : False)

OK : are_compatible_frequencies(a, b) == validate(a) and validate(b), sans logique conjointe supplémentaire


## 10 - Fonctions de commodité (`tsforecast/utils/frequency/utils.py`)

Ce module expose une instance globale partagée `_normalizer = FrequencyNormalizer()`
et des fonctions au niveau module qui délèguent directement à cette instance :
`to_literal`, `to_code`, `to_pandas_freq`, `to_dateoffset`, `is_higher_frequency`,
`validate_frequency`. Deux fonctions vont plus loin :
- `normalize_frequency()`, qui ajoute un paramètre `return_format` absent de la
  classe (section 10.1) ;
- `get_frequency_order()`, qui accède directement à un attribut privé de la classe
  (section 10.3).

`convert_frequency()` (également définie dans ce module) délègue entièrement à
`FrequencyConverter` : elle est hors périmètre ici (voir `frequency_converter.ipynb`).

### 10.1 - `normalize_frequency()` : le paramètre `return_format`

Contrairement à `FrequencyNormalizer.normalize()` (qui ne renvoie que la base),
`normalize_frequency()` offre 4 niveaux de détail via `return_format` :
- `'base'` (défaut) : équivalent à `normalizer.normalize()`
- `'with_position'` : base + position si identifiable (ex. `'QE'`)
- `'full'` : validation seulement, renvoie la chaîne d'entrée **inchangée**
- `'components'` : tuple `(base, position, suffixe)`

In [23]:
from tsforecast.utils.frequency.utils import normalize_frequency

exemples = ['QE-DEC', 'MS', 'D', 'monthly']
for fmt in ['base', 'with_position', 'full', 'components']:
    print(f"--- return_format={fmt!r} ---")
    for ex in exemples:
        r = normalize_frequency(ex, return_format=fmt)
        print(f"  {ex:10s} -> {r!r}")
    print()

# 'base' est équivalent à normalizer.normalize()
for ex in exemples:
    assert normalize_frequency(ex, return_format='base') == normalizer.normalize(ex)
print("OK : return_format='base' (défaut) == FrequencyNormalizer.normalize()")

--- return_format='base' ---
  QE-DEC     -> 'Q'
  MS         -> 'M'
  D          -> 'D'
  monthly    -> 'M'

--- return_format='with_position' ---
  QE-DEC     -> 'QE'
  MS         -> 'MS'
  D          -> 'D'
  monthly    -> 'M'

--- return_format='full' ---
  QE-DEC     -> 'QE-DEC'
  MS         -> 'MS'
  D          -> 'D'
  monthly    -> 'monthly'

--- return_format='components' ---
  QE-DEC     -> ('Q', 'E', 'DEC')
  MS         -> ('M', 'S', None)
  D          -> ('D', None, None)
  monthly    -> ('M', None, None)

OK : return_format='base' (défaut) == FrequencyNormalizer.normalize()


#### Piège : `return_format='full'` ne normalise PAS, elle valide puis renvoie l'entrée telle quelle

`normalize_frequency('monthly', return_format='full')` renvoie `'monthly'`, pas
`'M'` : la branche `'full'` du code utilise `parse_frequency`/`normalize()`
uniquement pour **valider** que la chaîne est acceptable, puis renvoie
systématiquement `frequency` (l'argument d'origine, non transformé) — y compris
pour un littéral qui n'a rien d'une chaîne pandas "complète".

In [24]:
r_full = normalize_frequency('monthly', return_format='full')
print("normalize_frequency('monthly', return_format='full') =", repr(r_full))
assert r_full == 'monthly'  # inchangé, PAS 'M'

# La validation a bien lieu : une chaîne invalide lève toujours une ValueError
try:
    normalize_frequency('xyz', return_format='full')
except ValueError as e:
    print("normalize_frequency('xyz', return_format='full') -> ValueError :", e)

normalize_frequency('monthly', return_format='full') = 'monthly'
normalize_frequency('xyz', return_format='full') -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']


#### `return_format='components'` : repli pour les littéraux non parsables

`parse_frequency()` attend une chaîne de fréquence pandas (majuscules), pas un nom
littéral en minuscules (`'monthly'`) : elle lève une `ValueError`, capturée en
interne pour retomber sur `normalizer.normalize(frequency)` avec `position=None` et
`suffix=None`. Un littéral ne peut donc jamais renvoyer de position/suffixe, même
si la fréquence sous-jacente en a implicitement un dans l'esprit de l'utilisateur.

In [25]:
print("components('MS')      =", normalize_frequency('MS', return_format='components'))
print("components('monthly') =", normalize_frequency('monthly', return_format='components'))
assert normalize_frequency('monthly', return_format='components') == ('M', None, None)

# Erreur invalide, propagée par le repli sur normalize()
try:
    normalize_frequency('xyz', return_format='components')
except ValueError as e:
    print("components('xyz') -> ValueError :", e)

# return_format invalide
try:
    normalize_frequency('D', return_format='bogus')
except ValueError as e:
    print("return_format='bogus' -> ValueError :", e)

components('MS')      = ('M', 'S', None)
components('monthly') = ('M', None, None)
components('xyz') -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']
return_format='bogus' -> ValueError : Invalid return_format: bogus. Must be one of: 'base', 'with_position', 'full', 'components'


### 10.2 - Délégation directe : équivalence avec une instance locale de `FrequencyNormalizer`

In [26]:
from tsforecast.utils.frequency.utils import (
    to_literal as fn_to_literal, to_code as fn_to_code, to_pandas_freq as fn_to_pandas_freq,
    to_dateoffset as fn_to_dateoffset, is_higher_frequency as fn_is_higher_frequency,
    validate_frequency,
)

for valeur in codes_supportes + litteraux_supportes:
    assert fn_to_literal(valeur) == normalizer.to_literal(valeur)
    assert fn_to_code(valeur) == normalizer.to_code(valeur)
    assert fn_to_pandas_freq(valeur) == normalizer.to_pandas_freq(valeur)
    assert fn_to_dateoffset(valeur) == normalizer.to_dateoffset(valeur)
    assert validate_frequency(valeur) == normalizer.validate(valeur)

for a, b in [('daily', 'monthly'), ('Q', 'W'), ('Y', 'Y')]:
    assert fn_is_higher_frequency(a, b) == normalizer.is_higher_frequency(a, b)

print("OK : les fonctions de utils.py renvoient exactement les mêmes résultats qu'une instance dédiée de FrequencyNormalizer")

OK : les fonctions de utils.py renvoient exactement les mêmes résultats qu'une instance dédiée de FrequencyNormalizer


### 10.3 - `get_frequency_order()` : expose directement un attribut "privé" (`_frequency_order`)

`get_frequency_order()` ne passe pas par une méthode publique de
`FrequencyNormalizer` mais lit directement `_normalizer._frequency_order.get(base_freq, 0)`
— un attribut préfixé `_`, donc a priori interne à la classe. C'est un point de
couplage à garder en tête : toute évolution du nom ou de la structure de cet
attribut dans `FrequencyNormalizer` casserait silencieusement cette fonction sans
qu'aucune API publique de la classe ne le signale. Le repli `.get(base_freq, 0)`
n'est cependant pas mort ici : contrairement à `is_higher_frequency()`,
`get_frequency_order()` normalise d'abord via `normalize_frequency()` (qui peut
lever une erreur), mais toute fréquence qui passe cette étape est par construction
déjà présente dans `_frequency_order` (mêmes mappings) -- le repli reste donc
inatteignable en pratique, pour la même raison que pour `is_higher_frequency()`.

In [27]:
from tsforecast.utils.frequency.utils import get_frequency_order

for c in codes_supportes:
    print(f"{c:4s} -> ordre {get_frequency_order(c)}")

print()
# Cohérence avec is_higher_frequency : get_frequency_order(a) < get_frequency_order(b) <=> is_higher_frequency(a, b)
for a in codes_supportes:
    for b in codes_supportes:
        assert (get_frequency_order(a) < get_frequency_order(b)) == normalizer.is_higher_frequency(a, b)
print("OK : get_frequency_order est cohérent avec is_higher_frequency sur toutes les paires de codes")

# Fréquence inconnue : ValueError propagée par normalize_frequency(), pas de repli silencieux sur 0
try:
    get_frequency_order('xyz')
except ValueError as e:
    print("\nget_frequency_order('xyz') -> ValueError :", e)

ns   -> ordre 1
us   -> ordre 2
ms   -> ordre 3
s    -> ordre 4
min  -> ordre 5
h    -> ordre 6
D    -> ordre 7
B    -> ordre 7.5
W    -> ordre 8
SM   -> ordre 8.5
M    -> ordre 9
Q    -> ordre 10
Y    -> ordre 11

OK : get_frequency_order est cohérent avec is_higher_frequency sur toutes les paires de codes

get_frequency_order('xyz') -> ValueError : Unsupported frequency: xyz. Supported frequencies: ['nanosecond', 'microsecond', 'millisecond', 'second', 'minute', 'hourly', 'daily', 'business_daily', 'weekly', 'semi_monthly', 'monthly', 'quarterly', 'annual'] or pandas codes: ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']


## 11 - Application à des index pandas réels détectés

On relie ici `normalize_frequency()`/`to_literal()` à des fréquences réellement
**détectées** sur des index pandas (via `detect_index_frequency`, plutôt que
saisies à la main), pour illustrer leur usage dans le pipeline du package.

In [28]:
from tsforecast.frequency.detector import detect_index_frequency

# Trois index à fréquences et positions différentes, comme dans un jeu de données
# macroéconomique réel (PIB trimestriel en position S, indicateurs mensuels en
# position E, balance commerciale annuelle en position E avec ancrage décembre)
idx_qs = pd.date_range('2020-01-01', periods=8, freq='QS')
idx_me = pd.date_range('2020-01-31', periods=12, freq='ME')
idx_ye = pd.date_range('2020-12-31', periods=5, freq='YE')

for idx, label in [(idx_qs, 'pib_trimestriel (QS)'), (idx_me, 'inflation_ipc (ME)'), (idx_ye, 'balance_commerciale_annuelle (YE)')]:
    detected = detect_index_frequency(index=idx, return_format='full')
    composantes = normalize_frequency(detected, return_format='components')
    print(f"{label:34s} détecté={detected!r:10s} -> components={composantes} -> littéral={normalizer.to_literal(composantes[0])}")

pib_trimestriel (QS)               détecté='QS-OCT'   -> components=('Q', 'S', 'OCT') -> littéral=quarterly
inflation_ipc (ME)                 détecté='ME'       -> components=('M', 'E', None) -> littéral=monthly
balance_commerciale_annuelle (YE)  détecté='YE-DEC'   -> components=('Y', 'E', 'DEC') -> littéral=annual


## 12 - Application aux indicateurs macroéconomiques (fréquences de publication)

On reprend les fréquences de publication définies dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (section 2) et
`duration_normalizer.ipynb` (section 9) : PIB trimestriel, inflation/chômage
mensuels, balance commerciale annuelle, dépenses publiques annuelles (FR/IT) ou
trimestrielles (DE).

In [29]:
indicateurs = {
    "pib_trimestriel": "Q",
    "inflation_ipc": "M",
    "taux_chomage": "M",
    "balance_commerciale_annuelle": "Y",
    "depenses_publiques_pib (FR/IT)": "Y",
    "depenses_publiques_pib (DE)": "Q",
}

# Nom littéral de chaque fréquence de publication
for nom, freq in indicateurs.items():
    print(f"{nom:32s} fréquence={freq} ({normalizer.to_literal(freq)})")

pib_trimestriel                  fréquence=Q (quarterly)
inflation_ipc                    fréquence=M (monthly)
taux_chomage                     fréquence=M (monthly)
balance_commerciale_annuelle     fréquence=Y (annual)
depenses_publiques_pib (FR/IT)   fréquence=Y (annual)
depenses_publiques_pib (DE)      fréquence=Q (quarterly)


In [30]:
# La fréquence de publication des dépenses allemandes (trimestrielle) est-elle
# plus fine que celle de la France/Italie (annuelle) ? -- confirme l'hétérogénéité
# de fréquence de publication introduite dans le notebook 3 pour cette variable
freq_de = indicateurs["depenses_publiques_pib (DE)"]
freq_fr_it = indicateurs["depenses_publiques_pib (FR/IT)"]
print(f"DE ({normalizer.to_literal(freq_de)}) plus fine que FR/IT ({normalizer.to_literal(freq_fr_it)}) :",
      normalizer.is_higher_frequency(freq_de, freq_fr_it))

# Comparaison de tous les indicateurs deux à deux via la fréquence la plus fine
plus_fin = min(indicateurs.items(), key=lambda kv: normalizer._frequency_order[kv[1]])
plus_grossier = max(indicateurs.items(), key=lambda kv: normalizer._frequency_order[kv[1]])
print(f"\nIndicateur à la fréquence la plus fine     : {plus_fin[0]} ({normalizer.to_literal(plus_fin[1])})")
print(f"Indicateur à la fréquence la plus grossière : {plus_grossier[0]} ({normalizer.to_literal(plus_grossier[1])})")

DE (quarterly) plus fine que FR/IT (annual) : True

Indicateur à la fréquence la plus fine     : inflation_ipc (monthly)
Indicateur à la fréquence la plus grossière : balance_commerciale_annuelle (annual)


## 13 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/frequency/test_normalizer.py` :

- **Écart entre les types déclarés et les mappings réels** : `FrequencyType`
  déclare `'T'` et `'A'` (alias pandas historiques de la minute et de l'année) qui
  ne sont PAS dans `_pandas_to_literal` -- leur normalisation échoue toujours
  (`ValueError`), y compris via le repli `parse_frequency` (garde-fou anti-boucle :
  la base extraite égale la valeur d'entrée). Inversement, `UserFrequencyType` ne
  déclare que 6 littéraux alors que la classe en supporte 13 (ex. `'nanosecond'`,
  `'minute'`, `'semi_monthly'` fonctionnent mais ne sont pas dans le type).
- **Identité** : `normalize(code) == code` et `to_code(code) == code` pour tous les
  codes réellement supportés (issus de `_pandas_to_literal`, pas de `FrequencyType`).
- **Bijection** : `to_code(to_literal(code)) == code` pour tous les codes supportés.
- **`to_code()` et `to_pandas_freq()` sont deux alias stricts de `normalize()`**
  (mêmes résultats, mêmes erreurs, sur toutes les entrées y compris invalides).
- **Sensibilité à la casse** : aucune normalisation de casse n'est appliquée --
  `'d'`, `'MONTHLY'`, `'MIN'` sont rejetés bien que `'D'`, `'monthly'`, `'min'`
  soient valides.
- **Repli `parse_frequency`** : les chaînes de fréquence pandas complètes (`'MS'`,
  `'QE-DEC'`, `'W-MON'`...) sont acceptées ; la position (S/E) et le suffixe
  d'ancrage sont ignorés, seule la base compte.
- **Types non `str`** : `normalize()`/`to_code()`/`to_literal()`/`to_pandas_freq()`/
  `to_dateoffset()` lèvent toujours `ValueError` (pas `TypeError`), y compris pour
  `None`, un `int`, une `list`.
- **`validate()` ne lève jamais d'exception** : renvoie `True`/`False` pour
  n'importe quel type d'entrée, y compris des types non hashables comme un `dict`,
  et pour les codes "fantômes" `'T'`/`'A'`.
- **`to_dateoffset()` perd la position** : `to_dateoffset('MS')`, `to_dateoffset('ME')`
  et `to_dateoffset('M')` renvoient le même `DateOffset` -- la normalisation
  préalable (qui ne garde que la base) empêche toute distinction start/end.
- **`is_higher_frequency()` est une comparaison stricte** : toujours `False` pour
  `is_higher_frequency(x, x)`.
- **Ordre des fréquences "spéciales"** : `D > B > W > SM > M` en granularité (`B`
  et `SM` ont un ordre non entier, `7.5` et `8.5`, pour s'insérer strictement entre
  deux fréquences standards).
- **`is_higher_frequency()`/`get_frequency_order()` propagent l'erreur sur
  fréquence invalide** : le repli `_frequency_order.get(code, 0)` est du code mort
  avec les mappings actuels, car `to_code()`/`normalize_frequency()` lèvent déjà
  une `ValueError` avant d'atteindre ce `.get()`.
- **`are_compatible_frequencies()` ne vérifie rien de conjoint** : strictement
  équivalent à `validate(freq1) and validate(freq2)` -- aucune notion de rapport de
  conversion raisonnable entre les deux fréquences n'est testée (ex. `'ns'`/`'Y'`
  sont jugées "compatibles").
- **`normalize_frequency()` (`utils.py`)** : `return_format='base'` équivaut à
  `FrequencyNormalizer.normalize()` ; `'with_position'` ajoute la position si
  identifiable ; `'components'` renvoie `(base, position, suffixe)` avec repli
  `(base, None, None)` pour les littéraux (non parsables par `parse_frequency`,
  qui n'accepte que des chaînes majuscules) ; **`'full'` NE normalise PAS** --
  elle valide seulement la chaîne d'entrée puis la renvoie inchangée (piège :
  `normalize_frequency('monthly', return_format='full') == 'monthly'`, pas `'M'`).
- **Fonctions de délégation de `utils.py`** (`to_literal`, `to_code`,
  `to_pandas_freq`, `to_dateoffset`, `is_higher_frequency`, `validate_frequency`) :
  résultats identiques à une instance locale de `FrequencyNormalizer`.
- **`get_frequency_order()`** : accède directement à l'attribut privé
  `_normalizer._frequency_order` plutôt qu'à une méthode publique de
  `FrequencyNormalizer` -- point de couplage fragile à surveiller si la classe
  évolue.
- **`convert_frequency()` de `utils.py`** : délègue entièrement à
  `FrequencyConverter.convert()`, hors périmètre de `FrequencyNormalizer` -- déjà
  couvert par `frequency_converter.ipynb`.